In [5]:
import cv2
import numpy as np
import os

# 1. SETUP
video_path = 'Video03.mp4'
output_dir = 'SUBMISSION_STAGES3'
os.makedirs(output_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    h, w = frame.shape[:2]
    
    # --- STAGE 1: GRAYSCALE ---
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # --- STAGE 2: FILTERING (CLAHE + HLS Mask) ---
    # Apply CLAHE to the grayscale for contrast
    enhanced_gray = clahe.apply(gray)
    
    # Apply HLS Masking for color isolation
    hls = cv2.cvtColor(frame, cv2.COLOR_BGR2HLS)
    white_mask = cv2.inRange(hls, np.array([0, 170, 0]), np.array([180, 255, 255]))
    
    # Apply ROI
    roi_mask = np.zeros_like(white_mask)
    pts = np.array([[(int(w*0.1), h), (int(w*0.4), int(h*0.6)), 
                     (int(w*0.6), int(h*0.6)), (int(w*0.9), h)]])
    cv2.fillPoly(roi_mask, pts, 255)
    filtered_result = cv2.bitwise_and(white_mask, roi_mask)

    # --- STAGE 3: DETECTION LOGIC ---
    edges = cv2.Canny(filtered_result, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, 50, minLineLength=30, maxLineGap=100)
    
    # Check if vanished
    is_vanished = lines is None
    
    # --- STAGE 4: FINAL DISPLAY ---
    final_display = frame.copy()
    if is_vanished:
        cv2.putText(final_display, "VANISHED", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0,0,255), 3)
        
        # SAVE ALL STAGES FOR SUBMISSION
        prefix = f"frame_{frame_count}_"
        cv2.imwrite(os.path.join(output_dir, prefix + "1_Original.jpg"), frame)
        cv2.imwrite(os.path.join(output_dir, prefix + "2_Grayscale.jpg"), gray)
        cv2.imwrite(os.path.join(output_dir, prefix + "3_FilteredMask.jpg"), filtered_result)
        cv2.imwrite(os.path.join(output_dir, prefix + "4_FinalDetection.jpg"), final_display)
        print(f"Vanished frame {frame_count} stages saved.")

    cv2.imshow('Processing...', final_display)
    frame_count += 1
    if cv2.waitKey(100) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

Vanished frame 93 stages saved.
Vanished frame 94 stages saved.
Vanished frame 95 stages saved.
Vanished frame 96 stages saved.
Vanished frame 97 stages saved.
Vanished frame 98 stages saved.
